# Kalman V3.2 — vectorbt Parity Patch V2

이 버전은 기존 parity Colab과 output folder를 재사용하지 않습니다.

- 모든 vectorbt 입력을 owned/writable NumPy array로 강제
- pandas 3.0.5 / vectorbt 1.1.0 버전 명시
- BUY/SELL signal = next-bar open
- MAX_HOLD = current-bar close
- size_type = percent
- 실패 시 전체 stdout/traceback을 Drive에 저장

**Research only / Toss OFF / Neon write OFF**


In [ ]:
from google.colab import drive
import json
import shutil
import subprocess
import sys
import traceback
from datetime import datetime
from pathlib import Path

PINNED_SHA = "563c3b333c9f83aac24af5069267511ce4e4669c"
SOURCE_BRANCH = "feature/historical-v3-2-portfolio-validation-20260913"
V1_RUN_TAG = "20260913_042850"
V3_CANDIDATE_TAG = "20260913_return_regime_v3_001"
PATCH_TAG = "20260913_v3_2_vectorbt_parity_v2_001"

drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = Path('/content/drive/MyDrive')


def locate_v1(root, tag):
    for p in [
        root / 'Market_Model_V2' / 'historical_quant_2017_v1' / tag,
        root / 'Kalman' / 'Market_Model_V2' / 'historical_quant_2017_v1' / tag,
    ]:
        if (p / 'historical_resume_complete.json').exists():
            return p
    raise FileNotFoundError(tag)


def locate_v3(root, tag):
    for p in [
        root / 'Market_Model_V2' / 'historical_quant_2017_v3_candidate' / tag,
        root / 'Kalman' / 'Market_Model_V2' / 'historical_quant_2017_v3_candidate' / tag,
    ]:
        q = p / 'historical_v3_candidate_summary.json'
        if q.exists():
            payload = json.loads(q.read_text(encoding='utf-8'))
            if payload.get('status') == 'COMPLETE':
                return p
    raise FileNotFoundError(tag)


repo = Path('/content/Codex')
if repo.exists():
    shutil.rmtree(repo)

v1 = locate_v1(DRIVE_ROOT, V1_RUN_TAG)
matrix_dir = v1.parents[1] / 'historical_matrices_v1'
v3_root = locate_v3(DRIVE_ROOT, V3_CANDIDATE_TAG)
out_root = (
    v1.parents[1]
    / 'historical_quant_2017_v3_2_vectorbt_patch'
    / PATCH_TAG
)
out_root.mkdir(parents=True, exist_ok=True)
full_log = out_root / 'parity_full_log.txt'
failure_json = out_root / 'parity_failure.json'


def run(cmd, *, cwd=None):
    args = [str(x) for x in cmd]
    line0 = "\n$ " + " ".join(args) + "\n"
    print(line0, end='')
    with full_log.open('a', encoding='utf-8') as log:
        log.write(line0)
        proc = subprocess.Popen(
            args,
            cwd=str(cwd) if cwd else None,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, args)


try:
    full_log.write_text(
        f"started_at={datetime.now().astimezone().isoformat()}\n"
        f"pinned_sha={PINNED_SHA}\n",
        encoding='utf-8',
    )

    run([
        'git', 'clone', '--branch', SOURCE_BRANCH,
        'https://github.com/kimtk94/Codex.git', repo,
    ])
    run(['git', '-C', repo, 'checkout', '--detach', PINNED_SHA])
    checked = subprocess.check_output(
        ['git', '-C', repo, 'rev-parse', 'HEAD'],
        text=True,
    ).strip()
    assert checked == PINNED_SHA, (checked, PINNED_SHA)

    app = repo / 'kalman-toss-gateway'
    module = app / 'research' / 'quant_stack' / 'historical_v3_2_portfolio_validation.py'
    test_file = app / 'tests' / 'test_historical_v3_2_portfolio_validation.py'
    spec_path = app / 'config' / 'model-v3-historical-return-regime-spec.json'

    if shutil.which('uv') is None:
        run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
    uv = shutil.which('uv')
    assert uv

    venv = Path('/content/.venv-kalman-v3-2-parity-v2')
    if venv.exists():
        shutil.rmtree(venv)
    run([uv, 'venv', venv])
    vpy = venv / 'bin' / 'python'

    run([
        uv, 'pip', 'install', '--python', vpy,
        'pandas==3.0.5',
        'numpy>=2.4.6',
        'pyarrow',
        'scipy<1.18',
        'scikit-learn',
        'PyPortfolioOpt==1.6.0',
        'riskfolio-lib==7.3.0',
        'vectorbt==1.1.0',
        'plotly<7',
        'pytest',
    ])
    run([uv, 'pip', 'check', '--python', vpy])
    run([
        vpy, '-c',
        (
            "import pandas as pd, numpy as np, vectorbt as vbt; "
            "print('pandas', pd.__version__); "
            "print('numpy', np.__version__); "
            "print('vectorbt', vbt.__version__)"
        ),
    ])
    run([vpy, '-m', 'py_compile', module, test_file])
    run([
        vpy, '-m', 'pytest', '-q',
        'tests/test_historical_v3_2_portfolio_validation.py',
    ], cwd=app)

    runner = Path('/content/run_vectorbt_parity_v2.py')
    runner.write_text(
        f"""
import json
from pathlib import Path
from research.quant_stack.historical_v3_2_portfolio_validation import run_vectorbt_validation_v32

v3_root = Path({str(v3_root)!r})
matrix_dir = Path({str(matrix_dir)!r})
spec = json.loads(Path({str(spec_path)!r}).read_text(encoding='utf-8'))
out_root = Path({str(out_root)!r})

result = run_vectorbt_validation_v32(
    v3_root=v3_root,
    matrix_dir=matrix_dir,
    spec=spec,
    output_root=out_root,
)
(out_root / 'vectorbt_patch_summary.json').write_text(
    json.dumps(result, ensure_ascii=False, indent=2, default=str) + '\\n',
    encoding='utf-8',
)
print(json.dumps(result, ensure_ascii=False, indent=2, default=str))
""",
        encoding='utf-8',
    )

    run([vpy, runner], cwd=app)

    summary = out_root / 'vectorbt_patch_summary.json'
    assert summary.exists(), summary
    print('\n' + '=' * 80)
    print('VECTORBT V3.2 PARITY V2 COMPLETE')
    print('=' * 80)
    print(summary.read_text(encoding='utf-8'))

except Exception as exc:
    payload = {
        'status': 'FAIL',
        'updated_at': datetime.now().astimezone().isoformat(),
        'pinned_sha': PINNED_SHA,
        'error_type': type(exc).__name__,
        'error': str(exc),
        'traceback': traceback.format_exc(),
        'full_log': str(full_log),
    }
    failure_json.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
    print('\nFAILURE SAVED:', failure_json)
    print(payload['traceback'])
    raise
